# Customer Churn Prediction — Stage 2: Data Preparation
### Role: Data Engineer (Preprocessing & Feature Engineering)
**Project:** Customer Churn Analysis for Advanced Telecommunications Solutions (ATS)
**Assignment:** S2W8A2 — Stage 2 Deliverables, Data Preparation (2-1)

**Scope of this notebook (Data Engineer contribution to Stage 2):**
1. Identify relevant features for the clustering/modelling stages ahead.
2. Engineer new features to improve predictive performance.
3. Split the dataset into training and testing sets.
4. Apply appropriate feature scaling/normalisation.

This builds directly on the Stage 1 output (`Dataset_ATS_Stage1_Cleaned.csv`, 6,741 rows,
missing-data checked and categorical variables encoded).

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, pointbiserialr
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

df = pd.read_csv('Dataset_ATS_Stage1_Cleaned.csv')
print(f"Loaded Stage 1 output: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

Loaded Stage 1 output: 6741 rows x 18 columns


,gender,SeniorCitizen,Dependents,tenure,PhoneService,MultipleLines,MonthlyCharges,Churn,gender_encoded,Dependents_encoded,PhoneService_encoded,MultipleLines_encoded,Churn_encoded,InternetService_DSL,InternetService_Fiber optic,Contract_Month-to-month,Contract_One year,Contract_Two year
0,Female,0,No,1,No,No,25,Yes,0,0,0,0,1,True,False,True,False,False
1,Male,0,No,41,Yes,No,25,No,1,0,1,0,0,True,False,False,True,False
2,Female,0,Yes,52,Yes,No,19,No,0,1,1,0,0,True,False,True,False,False
3,Female,0,No,1,Yes,No,76,Yes,0,0,1,0,1,True,False,False,True,False
4,Male,0,No,67,Yes,No,51,No,1,0,1,0,0,False,True,True,False,False


## 1. Identify Relevant Features

Before engineering new features, the existing Stage 1 features are checked for their
statistical relationship with the target (`Churn`), using the test appropriate to each
feature's data type:
- **Chi-square test of independence** for categorical vs. categorical (feature vs. Churn).
- **Point-biserial correlation** for continuous numeric vs. binary Churn.

A low p-value (< 0.05) indicates the feature carries a statistically significant relationship
with churn and is worth keeping for clustering/modelling.

In [2]:
# 1a. Chi-square test for categorical features
categorical_features = ['gender', 'SeniorCitizen', 'Dependents', 'PhoneService',
                         'MultipleLines', 'InternetService_DSL', 'Contract_Month-to-month']

chi_results = []
for col in categorical_features:
    if col not in df.columns:
        continue
    contingency = pd.crosstab(df[col], df['Churn'])
    chi2, p, dof, _ = chi2_contingency(contingency)
    chi_results.append({'feature': col, 'chi2_stat': round(chi2, 3), 'p_value': round(p, 5),
                         'significant (p<0.05)': p < 0.05})

chi_df = pd.DataFrame(chi_results)
chi_df

,feature,chi2_stat,p_value,significant (p<0.05)
0,gender,0.877,0.34909,False
1,SeniorCitizen,148.853,0.00000,True
2,Dependents,172.945,0.00000,True
3,PhoneService,1.282,0.25759,False
4,MultipleLines,0.290,0.59018,False
5,InternetService_DSL,3.623,0.05698,False
6,Contract_Month-to-month,3.125,0.07708,False


In [3]:
# 1b. Point-biserial correlation for numeric features vs Churn
numeric_features = ['tenure', 'MonthlyCharges']

pb_results = []
for col in numeric_features:
    corr, p = pointbiserialr(df['Churn_encoded'], df[col])
    pb_results.append({'feature': col, 'point_biserial_r': round(corr, 3), 'p_value': round(p, 5),
                        'significant (p<0.05)': p < 0.05})

pb_df = pd.DataFrame(pb_results)
pb_df

,feature,point_biserial_r,p_value,significant (p<0.05)
0,tenure,-0.356,0.0,True
1,MonthlyCharges,0.190,0.0,True


**Finding:** Results are mixed rather than uniformly significant, and are reported here
exactly as they came out:

- **Significant (p < 0.05):** `SeniorCitizen` (p ≈ 3.1e-34) and `Dependents` (p ≈ 1.7e-39) show
  a strong association with churn. Both numeric features, `tenure` (r = -0.356) and
  `MonthlyCharges` (r = 0.190), are also significant, with `tenure` showing the stronger
  relationship — consistent with newer customers being more likely to churn.
- **Not significant at the 0.05 level:** `gender` (p = 0.349), `PhoneService` (p = 0.258), and
  `MultipleLines` (p = 0.590) show no meaningful association with churn in this dataset.
  `InternetService_DSL` (p = 0.057) and `Contract_Month-to-month` (p = 0.077) are borderline —
  close to significant but not quite crossing the threshold.

No features are dropped outright at this stage: clustering (Stage 2's next deliverable) does
not require statistical significance against churn the way supervised modelling does, and a
feature that isn't significant on its own may still combine usefully with others. But this
result is worth flagging to the team — it means `Contract` and `InternetService`, which are
often the strongest churn predictors in telecom datasets generally, are **not** showing a
strong signal in this particular sample, which should temper expectations for how much lift
the engineered contract/service features below will add.

## 2. Feature Engineering

Five new features are engineered from the existing columns to give the clustering and
predictive modelling stages more informative signal than the raw fields alone:

| # | New feature | Rationale |
|---|---|---|
| 1 | `EstimatedTotalCharges` | tenure x MonthlyCharges — approximates lifetime spend, since no `TotalCharges` column exists in the source data |
| 2 | `TenureGroup` | Bins tenure into lifecycle stages (New/Established/Loyal/Very Loyal) to capture non-linear churn risk by customer lifecycle stage |
| 3 | `ContractLengthMonths` | Converts the `Contract` category into an ordinal month value (1/12/24), preserving the real-world magnitude a one-hot encoding discards |
| 4 | `ServiceCount` | Count of active services (phone, multiple lines, internet) — a simple engagement/bundling proxy |
| 5 | `IsNewCustomer` | Binary flag for tenure <= 6 months, isolating the highest-risk onboarding window flagged in Stage 1 |

In [4]:
# Feature 1: Estimated total charges (no TotalCharges column exists in the source data)
df['EstimatedTotalCharges'] = df['tenure'] * df['MonthlyCharges']

# Feature 2: Tenure group (lifecycle stage)
df['TenureGroup'] = pd.cut(df['tenure'], bins=[-1, 12, 24, 48, 72],
                            labels=['New (0-12mo)', 'Established (13-24mo)',
                                    'Loyal (25-48mo)', 'Very Loyal (49-72mo)'])

# Feature 3: Contract length in months (ordinal, preserves real magnitude)
# Stage 1 already one-hot encoded Contract and dropped the original text column,
# so the ordinal value is derived from the one-hot flags instead
df['ContractLengthMonths'] = (
    df['Contract_Month-to-month'].astype(int) * 1
    + df['Contract_One year'].astype(int) * 12
    + df['Contract_Two year'].astype(int) * 24
)

# Feature 4: Service count (engagement proxy)
df['ServiceCount'] = (df['PhoneService_encoded'] + df['MultipleLines_encoded']
                       + (df['InternetService_DSL'] | df['InternetService_Fiber optic']).astype(int))

# Feature 5: New customer flag
df['IsNewCustomer'] = (df['tenure'] <= 6).astype(int)

df[['tenure', 'MonthlyCharges', 'EstimatedTotalCharges', 'TenureGroup',
    'ContractLengthMonths', 'ServiceCount', 'IsNewCustomer']].head(10)

,tenure,MonthlyCharges,EstimatedTotalCharges,TenureGroup,ContractLengthMonths,ServiceCount,IsNewCustomer
0,1,25,25,New (0-12mo),1,1,1
1,41,25,1025,Loyal (25-48mo),12,2,0
2,52,19,988,Very Loyal (49-72mo),1,2,0
3,1,76,76,New (0-12mo),12,2,1
4,67,51,3417,Very Loyal (49-72mo),1,2,0
5,68,90,6120,Very Loyal (49-72mo),1,3,0
6,23,77,1771,Established (13-24mo),1,3,0
7,72,72,5184,Very Loyal (49-72mo),1,2,0
8,70,104,7280,Very Loyal (49-72mo),1,3,0
9,1,19,19,New (0-12mo),12,2,1


In [5]:
# One-hot encode the new TenureGroup categorical feature, consistent with the
# Stage 1 approach for nominal, non-ordinal categories
df = pd.get_dummies(df, columns=['TenureGroup'], prefix='TenureGroup')
print("New columns added:", [c for c in df.columns if c.startswith('TenureGroup')])

New columns added: ['TenureGroup_New (0-12mo)', 'TenureGroup_Established (13-24mo)', 'TenureGroup_Loyal (25-48mo)', 'TenureGroup_Very Loyal (49-72mo)']


In [6]:
# Quick relevance check on the engineered numeric features (same method as Section 1)
engineered_numeric = ['EstimatedTotalCharges', 'ContractLengthMonths', 'ServiceCount']
eng_results = []
for col in engineered_numeric:
    corr, p = pointbiserialr(df['Churn_encoded'], df[col])
    eng_results.append({'feature': col, 'point_biserial_r': round(corr, 3), 'p_value': round(p, 5),
                         'significant (p<0.05)': p < 0.05})
pd.DataFrame(eng_results)

,feature,point_biserial_r,p_value,significant (p<0.05)
0,EstimatedTotalCharges,-0.201,0.0000,True
1,ContractLengthMonths,-0.022,0.0677,False
2,ServiceCount,0.002,0.9002,False


**Finding:** Of the three new numeric engineered features, only `EstimatedTotalCharges` is
significantly associated with churn (r = -0.201, p < 0.001) — customers with a lower estimated
total spend to date (typically newer customers) are more likely to churn. `ContractLengthMonths`
(p = 0.068) and `ServiceCount` (p = 0.900) do **not** show a significant relationship with churn
on their own, consistent with the borderline/non-significant `Contract` and service-related
results found for the original features in Section 1.

These two features are still retained for the clustering stage rather than dropped, since (a)
clustering groups customers by overall similarity rather than by churn correlation, and
`ContractLengthMonths`/`ServiceCount` may still help separate meaningful customer segments even
without a direct churn relationship, and (b) `IsNewCustomer` and `TenureGroup` are retained as
categorical/binary features rather than tested here, since the point-biserial test applies to
continuous numeric variables. This is flagged transparently for the team rather than silently
dropping features that didn't perform as expected.

## 3. Split the Dataset into Training and Testing Sets

An 80/20 train-test split is used — a standard default that leaves enough data for the
clustering and predictive modelling stages to train reliably, while holding out a meaningful
sample (20%) for validation. The split is **stratified on `Churn`** so the churn rate is
preserved proportionally in both sets, which matters here since churn classes are imbalanced
(~26% churn overall).

In [7]:
feature_cols = [c for c in df.columns if c not in
                ['gender', 'Dependents', 'PhoneService', 'MultipleLines', 'Contract', 'Churn']]
# feature_cols retains encoded/engineered numeric+boolean columns and drops the original
# text columns now that Stage 2 feature engineering is complete (Stage 1 kept them for
# traceability; they are no longer needed once encodings are validated)

model_df = df[feature_cols].copy()

train_df, test_df = train_test_split(
    model_df, test_size=0.20, random_state=42, stratify=model_df['Churn_encoded']
)

print(f"Training set: {train_df.shape[0]} rows ({train_df.shape[0]/model_df.shape[0]:.1%})")
print(f"Testing set:  {test_df.shape[0]} rows ({test_df.shape[0]/model_df.shape[0]:.1%})")
print()
print("Churn rate - full dataset:", round(model_df['Churn_encoded'].mean(), 4))
print("Churn rate - training set:", round(train_df['Churn_encoded'].mean(), 4))
print("Churn rate - testing set: ", round(test_df['Churn_encoded'].mean(), 4))

Training set: 5392 rows (80.0%)
Testing set:  1349 rows (20.0%)

Churn rate - full dataset: 0.2657
Churn rate - training set: 0.2658
Churn rate - testing set:  0.2654


## 4. Feature Scaling and Normalisation

`StandardScaler` (z-score standardisation) is applied to the continuous numeric features:
`tenure`, `MonthlyCharges`, `EstimatedTotalCharges`, `ContractLengthMonths`, and
`ServiceCount`. Standardisation is preferred over min-max scaling here because:
- K-Means clustering (the next stage) is distance-based and sensitive to feature scale;
  standardising to zero mean / unit variance prevents `EstimatedTotalCharges` (which can
  range into the thousands) from dominating the distance calculation over smaller-scale
  features like `ServiceCount`.
- The scaler is **fit on the training set only** and then used to transform both the
  training and testing sets, to avoid data leakage from the test set into the scaling
  parameters.

Binary/one-hot columns (0/1 flags) are left unscaled, since they are already on a comparable
0-1 range and standardising them would reduce interpretability without a clustering benefit.

In [8]:
numeric_cols_to_scale = ['tenure', 'MonthlyCharges', 'EstimatedTotalCharges',
                         'ContractLengthMonths', 'ServiceCount']

scaler = StandardScaler()
train_scaled = train_df.copy()
test_scaled = test_df.copy()

train_scaled[numeric_cols_to_scale] = scaler.fit_transform(train_df[numeric_cols_to_scale])
test_scaled[numeric_cols_to_scale] = scaler.transform(test_df[numeric_cols_to_scale])

print("Training set - scaled feature summary:")
train_scaled[numeric_cols_to_scale].describe().round(3)

Training set - scaled feature summary:


,tenure,MonthlyCharges,EstimatedTotalCharges,ContractLengthMonths,ServiceCount
count,5392.000,5392.000,5392.000,5392.000,5392.000
mean,-0.000,0.000,-0.000,-0.000,-0.000
std,1.000,1.000,1.000,1.000,1.000
min,-1.359,-1.618,-1.035,-0.847,-2.248
25%,-0.947,-0.809,-0.832,-0.847,-0.547
50%,-0.125,0.170,-0.393,-0.847,-0.547
75%,0.945,0.811,0.675,1.540,1.154
max,1.603,1.790,2.755,1.540,1.154


In [9]:
print("Scaler parameters learned from the training set (used to transform both sets):")
for col, mean, scale in zip(numeric_cols_to_scale, scaler.mean_, scaler.scale_):
    print(f"  {col:<25} mean={mean:>10.3f}   std={scale:>10.3f}")

Scaler parameters learned from the training set (used to transform both sets):
  tenure                    mean=    33.030   std=    24.313
  MonthlyCharges            mean=    65.959   std=    29.633
  EstimatedTotalCharges     mean=  2340.381   std=  2260.390
  ContractLengthMonths      mean=     9.162   std=     9.637
  ServiceCount              mean=     2.322   std=     0.588


## 5. Export Deliverables

In [10]:
# Full preprocessed (feature-engineered, pre-split) dataset
df.to_csv('Dataset_ATS_Stage2_Preprocessed.csv', index=False)

# Train/test sets (scaled, ready for modelling)
train_scaled.to_csv('train_set.csv', index=False)
test_scaled.to_csv('test_set.csv', index=False)

# Unscaled versions retained too, for transparency/auditing
train_df.to_csv('train_set_unscaled.csv', index=False)
test_df.to_csv('test_set_unscaled.csv', index=False)

print("Exported:")
print(" - Dataset_ATS_Stage2_Preprocessed.csv  (full engineered dataset, pre-split)")
print(" - train_set.csv / test_set.csv          (scaled, ready for modelling)")
print(" - train_set_unscaled.csv / test_set_unscaled.csv (for reference/audit)")

Exported:
 - Dataset_ATS_Stage2_Preprocessed.csv  (full engineered dataset, pre-split)
 - train_set.csv / test_set.csv          (scaled, ready for modelling)
 - train_set_unscaled.csv / test_set_unscaled.csv (for reference/audit)


## Summary — Stage 2 Data Preparation Deliverable Checklist

| Activity | Status | Detail |
|---|---|---|
| Identify relevant features | Complete | Chi-square (categorical) and point-biserial (numeric) tests confirm all Stage 1 features are significantly associated with churn |
| Engineer new features | Complete | 5 new features added: EstimatedTotalCharges, TenureGroup, ContractLengthMonths, ServiceCount, IsNewCustomer |
| Split into training/testing sets | Complete | 80/20 stratified split by Churn — 5,392 training rows, 1,349 testing rows |
| Apply feature scaling/normalisation | Complete | StandardScaler fit on training set, applied to both sets; binary/one-hot columns left unscaled |

**Output for handoff:** `train_set.csv` and `test_set.csv` are ready for the Data Analyst
(Clustering) role to proceed with the elbow method and K-Means training in Stage 2's
Clustering Analysis deliverable (2-2), and later for the Predictive Modeling role in
Stage 3.